# Lab 4.10 &mdash; Fine-tune a Sentiment Classifier (before vs after)

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Establish an untrained baseline accuracy
- Fine-tune (train) the classifier head and beat the baseline
- Predict sentiment on brand-new sentences

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 4 slides &mdash; Fine-tuning end-to-end](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-04-10"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
This is the client's **fine-tune for sentiment**, in its portable form: a pre-trained feature
extractor (frozen) + a head we **train** on labelled reviews. We measure accuracy **before**
(an untrained majority baseline) vs **after** training, then predict on new text. The optional
cell does the *same task* by fine-tuning a real **bert-tiny** &mdash; the sandbox target.

In [ ]:
# DEMO
# A tiny labelled sentiment dataset (1 = positive, 0 = negative)
SENT = [
    ("i love this it is great", 1),
    ("a great and wonderful film", 1),
    ("truly wonderful i love it", 1),
    ("excellent and brilliant work", 1),
    ("the best most brilliant story", 1),
    ("i love how great it is", 1),
    ("wonderful excellent and great fun", 1),
    ("a brilliant and great success", 1),
    ("great fun i really love it", 1),
    ("the best film wonderful and brilliant", 1),
    ("excellent great and lovely work", 1),
    ("i love this brilliant great film", 1),
    ("wonderful great and the best", 1),
    ("so good i love it great", 1),
    ("i hate this it is terrible", 0),
    ("a terrible and awful film", 0),
    ("truly awful i hate it", 0),
    ("boring and terrible work", 0),
    ("the worst most boring story", 0),
    ("i hate how bad it is", 0),
    ("awful boring and dull mess", 0),
    ("a terrible and bad failure", 0),
    ("boring mess i really hate it", 0),
    ("the worst film awful and boring", 0),
    ("terrible bad and dull work", 0),
    ("i hate this awful boring film", 0),
    ("awful terrible and the worst", 0),
    ("so bad i hate it terrible", 0),
]
texts  = [t for t, y in SENT]
labels = [y for t, y in SENT]
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
Xtr_t, Xval_t, ytr, yval = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)
vec = TfidfVectorizer().fit(Xtr_t)
Xtr, Xval = vec.transform(Xtr_t), vec.transform(Xval_t)
print("ready: train", Xtr.shape[0], "val", Xval.shape[0])

## Your Turn
Compute the baseline, train the head, compare, and predict on new sentences.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from collections import Counter

def baseline_acc():
    # always predict the majority class of the training labels
    majority = Counter(ytr).most_common(1)[0][0]
    return ___   # TODO: accuracy of always predicting `majority` on yval

head = LogisticRegression(max_iter=1000)
def trained_acc():
    ___   # TODO: fit head on (Xtr, ytr)
    return accuracy_score(yval, head.predict(Xval))

def predict_new(sentences):
    head.fit(Xtr, ytr)
    return list(head.predict(___))   # TODO: vectorise sentences with vec.transform

try:
    print('baseline:', round(baseline_acc(),3), '| trained:', round(trained_acc(),3))
    print('predict:', predict_new(['a wonderful brilliant film', 'a boring awful mess']))
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("baseline computed (<= 0.7 on balanced data)", lambda: baseline_acc() <= 0.7)
expect_true("trained accuracy >= 0.8", lambda: trained_acc() >= 0.8)
expect_true("training beats the baseline", lambda: trained_acc() > baseline_acc())
expect_true("predicts new positive/negative correctly", lambda: predict_new(['a wonderful brilliant film','a boring awful mess']) == [1,0])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; the real thing with Hugging Face (not graded)
Fine-tune a REAL bert-tiny on the same sentences (a few CPU steps) and predict. Safe to skip &mdash; it needs `pip install transformers torch` and a one-time model
download. If `transformers` is not installed (or offline), the cell prints a note and moves on.
**Real BERT fine-tuning is the managed-sandbox target; the graded steps above run offline so the
lab always works.**

In [ ]:
try:
    import torch, numpy as np
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    name = "prajjwal1/bert-tiny"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4)
    enc = tok(Xtr_t, padding=True, truncation=True, return_tensors="pt")
    yt = torch.tensor(ytr)
    model.train()
    for step in range(30):                      # a few steps on tiny data
        opt.zero_grad()
        out = model(**enc, labels=yt)
        out.loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        venc = tok(Xval_t, padding=True, truncation=True, return_tensors="pt")
        pred = model(**venc).logits.argmax(-1).numpy()
    print("real bert-tiny val accuracy:", float((pred == np.array(yval)).mean()))
except Exception as e:
    print("Optional real-model cell skipped -- the graded head above already fine-tuned a model.")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
You fine-tuned a classifier: an untrained baseline, then a head trained on your labels that beats it and predicts new text. Real BERT fine-tuning (optional cell) is the same loop, scaled.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>